### pruning for CT reduction

In [1]:
import torch
from torch import nn

from torchvision.models import vgg16
import copy


from he_computation_estimation import *

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [4]:
MODEL_PATH="runs/vgg16-fp32-baseline/best.pt"
NUM_LABELS=100

train_state_dict = torch.load(MODEL_PATH, map_location="cpu", weights_only=True)

model = vgg16(weights=None)
#reduces final classifier to 100 since it defaults to 1000
model.classifier[-1] = nn.Linear(model.classifier[-1].in_features, NUM_LABELS) # type: ignore

model.load_state_dict(train_state_dict["model"])
# Move every layer together, including the newly created classifier.
model = model.to(device)
print(model)

# checkpoint = {"model": model.state_dict(), "optimizer": optimizer.state_dict(),
#               "scheduler": scheduler.state_dict(), "epoch": epoch + 1, "best_top1": best_top1,
#               "config": config, "classes": classes, "history": history, "environment": environment,
#               "python_rng": random.getstate(), "torch_rng": torch.get_rng_state(),
#               "loader_rng": generator.get_state(),
#               "cuda_rng": torch.cuda.get_rng_state_all() if device.type == "cuda" else []}

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [5]:
print(list(model.named_parameters()))


[('features.0.weight', Parameter containing:
tensor([[[[ 0.0837,  0.1574,  0.0393],
          [-0.0117,  0.0458, -0.0457],
          [ 0.0503,  0.0432,  0.1284]],

         [[-0.0149, -0.1612, -0.1344],
          [-0.0217, -0.0514, -0.1089],
          [-0.1314, -0.0095, -0.0126]],

         [[ 0.1124,  0.1111,  0.1175],
          [-0.0241, -0.0111,  0.0409],
          [ 0.0684, -0.0006,  0.0393]]],


        [[[ 0.0676,  0.1161, -0.0164],
          [ 0.0117,  0.0579,  0.0623],
          [-0.0198,  0.0773,  0.0116]],

         [[-0.0321,  0.0113, -0.0052],
          [-0.0744,  0.0108, -0.1419],
          [-0.0932,  0.0393,  0.0567]],

         [[ 0.0438,  0.0044, -0.0298],
          [-0.0147, -0.0418,  0.0209],
          [ 0.0113, -0.1137,  0.1034]]],


        [[[ 0.0235,  0.0151, -0.0242],
          [-0.0499, -0.0219, -0.0556],
          [-0.0255, -0.0636, -0.0716]],

         [[-0.0975, -0.0465, -0.0118],
          [ 0.0600, -0.0509,  0.0556],
          [-0.0600,  0.0186,  0.0304]],


In [6]:
import torch.nn.utils.prune as prune

### L1 channel pruning: select, then physically compact

For the usual im2col convention, `X` has shape `[K, L]`, where
`K = Cin * Kh * Kw` and `L = Hout * Wout` (ungrouped convolution).
Removing input channels reduces **K**, without changing the spatial dimensions.
To reduce the input of convolution B, prune output channels of its preceding
convolution A and remove the corresponding input-channel slices from B.

`prune.ln_structured(A, "weight", amount=ratio, n=1, dim=0)` ranks A's
`[Cout, Cin, Kh, Kw]` filters by their L1 norms and masks the smallest ones.
`dim=0` selects output filters; `dim=1` would mask A's own input-channel slices
without removing the channels supplied by the preceding layer.
Masking alone leaves tensor shapes and dense computation unchanged, including
after `prune.remove`: that only makes the masked weights permanent.
Zero the selected biases too, since zero weights with a nonzero bias still
produce a feature map. Then rebuild A and B with fewer channels.

Cell 8 demonstrates **one pair** in the original, non-BN torchvision VGG16.
It keeps VGG's original forward, including flatten before the classifier.
With `feature_index=0` and ratio 0.25, A has 64 → 48 output channels and B has
64 → 48 input channels, so B's im2col rows fall from 576 → 432.
ReLU and max pooling between convolutions preserve channel identities.
The helper also accepts other convolution indices, except the final convolution:
pruning that layer would additionally require slicing the first Linear layer's
input in channel-sized 7×7 blocks. Batch normalization, groups, and branches
would need additional dependency handling and are outside this example.

The masked and compact models are mathematically equivalent, but CUDA may use
different convolution algorithms and accumulation orders for their different
channel shapes. TF32 can make these differences larger. The demo temporarily
uses IEEE FP32 (disables TF32/autocast) and disables cuDNN benchmarking during
the comparison, prints the maximum/mean absolute logit error, and keeps the
`rtol=1e-4, atol=1e-4` assertion. Backend settings are restored afterward, including
when a check fails; the sweep retains its original precision settings.
See [PyTorch numerical accuracy](https://docs.pytorch.org/docs/stable/notes/numerical_accuracy.html).

**Ciphertext counts depend on the packing convention.** If each column of
`X[K, L]` is chunked into `S` slots, its count is `ceil(K / S)`, so a row reduction
only saves ciphertexts when it crosses a chunk boundary. For example, 576 → 432
rows still needs one ciphertext per column if `S >= 576`.
Our existing `model_to_he_im2col_ops` instead packs columns of **`X.T[L, K]`**:
for one sample/group its input count is `K * ceil(L / N)`. Here pruning reduces
the number of encrypted columns K; ciphertexts per column remain `ceil(L / N)`.
Use the actual estimator/packing scheme when choosing useful pruning ratios.

References: [PyTorch ln_structured](https://docs.pytorch.org/docs/stable/generated/torch.nn.utils.prune.ln_structured.html),
[pruning masks and remove](https://docs.pytorch.org/tutorials/intermediate/pruning_tutorial.html).


In [7]:
STRUCT_WEIGHT_RATIO = 0.25
PRUNE_FEATURE_INDEX = 0  # Try 2, 5, 7, ... for later convolution pairs.


from contextlib import contextmanager


@contextmanager
def compaction_check_precision(check_device):
    """Use IEEE FP32 for this check, then restore the caller's backend settings."""
    settings = []
    if check_device.type == "cuda":
        if (hasattr(torch.backends.cuda.matmul, "fp32_precision")
                and hasattr(torch.backends.cudnn, "conv")
                and hasattr(torch.backends.cudnn.conv, "fp32_precision")):
            settings = [(torch.backends.cuda.matmul, "fp32_precision", "ieee"),
                        (torch.backends.cudnn.conv, "fp32_precision", "ieee")]
        else:  # Compatibility with older PyTorch precision controls.
            settings = [(torch.backends.cuda.matmul, "allow_tf32", False),
                        (torch.backends.cudnn, "allow_tf32", False)]
        settings.append((torch.backends.cudnn, "benchmark", False))
    saved = [(backend, name, getattr(backend, name)) for backend, name, _ in settings]
    try:
        for backend, name, value in settings:
            setattr(backend, name, value)
        with torch.autocast(device_type=check_device.type, enabled=False):
            yield
    finally:
        for backend, name, value in reversed(saved):
            setattr(backend, name, value)


def prune_conv_pair(base, ratio, feature_index=0):
    """Return (compact VGG, bias-masked reference); leave base weights intact."""
    conv_indices = [i for i, layer in enumerate(base.features)
                    if isinstance(layer, nn.Conv2d)]
    if feature_index not in conv_indices[:-1]:
        raise ValueError("Choose a convolution with a following convolution")
    
    next_index = conv_indices[conv_indices.index(feature_index) + 1]
    a, b = base.features[feature_index], base.features[next_index]
    if not 0 <= ratio < 1 or round(ratio * a.out_channels) >= a.out_channels:
        raise ValueError("ratio must leave at least one output channel")
    if a.groups != 1 or b.groups != 1 or a.out_channels != b.in_channels:
        raise ValueError("This example requires matching, ungrouped convolutions")
    if any(not isinstance(layer, (nn.ReLU, nn.MaxPool2d))
           for layer in list(base.features)[feature_index + 1:next_index]):
        raise ValueError("Only ReLU/max pooling may intervene in this example")

    masked = copy.deepcopy(base).eval()
    a = masked.features[feature_index]
    prune.ln_structured(a, name="weight", amount=ratio, n=1, dim=0)
    alive = a.weight_mask.flatten(1).any(dim=1)
    keep = alive.nonzero(as_tuple=True)[0]
    with torch.no_grad():
        if a.bias is not None:
            a.bias.mul_(alive)  # Ensure removed channels really output zero.
    prune.remove(a, "weight")  # Still dense; no shape change at this point.

    compact = copy.deepcopy(masked)
    # Slice A's outputs and B's matching inputs, preserving channel order.
    for index in (feature_index, next_index):
        old = masked.features[index]
        inputs = (keep if index == next_index else
                  torch.arange(old.in_channels, device=old.weight.device))
        outputs = (keep if index == feature_index else
                   torch.arange(old.out_channels, device=old.weight.device))
        new = nn.Conv2d(
            len(inputs), len(outputs), old.kernel_size, stride=old.stride,
            padding=old.padding, dilation=old.dilation, bias=old.bias is not None,
            padding_mode=old.padding_mode, device=old.weight.device,
            dtype=old.weight.dtype,
        )
        with torch.no_grad():
            new.weight.copy_(old.weight[outputs][:, inputs])
            if old.bias is not None:
                new.bias.copy_(old.bias[outputs])
        compact.features[index] = new
    return compact.eval(), masked


l1_m, masked_reference = prune_conv_pair(model, STRUCT_WEIGHT_RATIO, PRUNE_FEATURE_INDEX)
conv_indices = [i for i, layer in enumerate(model.features) if isinstance(layer, nn.Conv2d)]
next_index = conv_indices[conv_indices.index(PRUNE_FEATURE_INDEX) + 1]
image_size = train_state_dict.get("config", {}).get("image_size", 224)
parameter = next(model.parameters())
example = torch.randn(1, 3, image_size, image_size,
                      device=parameter.device, dtype=parameter.dtype)
model.eval()  # Disable dropout for all forward comparisons.

with compaction_check_precision(parameter.device), torch.no_grad():
    baseline_logits = model(example)
    masked_logits = masked_reference(example)
    compact_logits = l1_m(example)
    before_shape = tuple(model.features[:next_index](example).shape)
    after_shape = tuple(l1_m.features[:next_index](example).shape)

# Different channel shapes can still change FP32 accumulation order.
compaction_error = (compact_logits - masked_logits).abs()
print(f"Compact vs masked logits: max absolute error={compaction_error.max().item():.6g}, "
      f"mean absolute error={compaction_error.mean().item():.6g}")
torch.testing.assert_close(compact_logits, masked_logits, rtol=1e-4, atol=1e-4)
print(f"features[{next_index}] input: {before_shape} -> {after_shape}")
for label, network in (("baseline", model), ("masked", masked_reference), ("compact", l1_m)):
    conv = network.features[next_index]
    rows = conv.in_channels * conv.kernel_size[0] * conv.kernel_size[1]
    parameters = sum(p.numel() for p in network.parameters())
    print(f"{label:8s}: im2col rows={rows}, parameters={parameters:,}")

print(f"Output shape: {tuple(compact_logits.shape)}; compact/masked outputs agree")
print(f"Mean |baseline - pruned logits| on random input: "
      f"{(baseline_logits - compact_logits).abs().mean().item():.6g}")
print("Random-input checks verify shapes and compaction, not classification accuracy.")
del masked_reference  # Keep the compact model l1_m for evaluation/fine-tuning.


Compact vs masked logits: max absolute error=0, mean absolute error=0
features[2] input: (1, 64, 224, 224) -> (1, 48, 224, 224)
baseline: im2col rows=576, parameters=134,670,244
masked  : im2col rows=576, parameters=134,670,244
compact : im2col rows=432, parameters=134,660,580
Output shape: (1, 100); compact/masked outputs agree
Mean |baseline - pruned logits| on random input: 0.311972
Random-input checks verify shapes and compaction, not classification accuracy.


### Ratio sweep and accuracy recovery

Run cells 10–11 after cell 8. The default ratios are **0.25, 0.30, 0.35, 0.40,
0.45, 0.50**. Each trial prunes the same convolution pair selected by
`PRUNE_FEATURE_INDEX`, starts from the original `model`, evaluates immediately,
and fine-tunes for three epochs with a fresh SGD optimizer at LR 0.001.
These training settings are editable starting points, not a tuned recovery recipe.

The sweep reuses the checkpoint's dataset, split seed, image size, preprocessing,
and class order. Training uses only the training split; the same held-out
validation split is used for the baseline, immediately pruned model, and every
fine-tuning epoch. The final comparison uses the **last epoch**, with the complete
epoch history retained. The official CIFAR-100 test split remains untouched.

Cell 10 exposes the dataset path and training settings. Its default data path
searches the kernel's working directory and its parents for the checkpoint's
relative data directory; check the printed path before running cell 11.
No dataset downloads occur. Training uses the notebook model's current device.
The default `num_workers=0` works in notebook kernels, and gradient accumulation
keeps the checkpoint's effective batch size when the batch-size settings match.

Cell 11 prints a comparison table and saves `results.json` (settings, baseline,
metrics, and epoch histories) and `results.csv` (one row per completed ratio)
under `SWEEP_OUTPUT_DIR`, updating them after each completed trial. Rerunning
cell 11 replaces these result files. The six fine-tuning runs can take substantial
time, especially on CPU; a synthetic smoke check is not an accuracy measurement.
The baseline weights remain unchanged, and trial models are released after use.
Add `0.0` to `SWEEP_RATIOS` if you also want an unpruned fine-tuning control.


In [8]:
from pathlib import Path

torch.backends.cudnn.benchmark = True
# torch.backends.cuda.matmul.fp32_precision = "tf32"
# torch.backends.cudnn.conv.fp32_precision = "tf32"

SWEEP_RATIOS = [percent / 100 for percent in range(20, 51, 10)]
# FINETUNE_EPOCHS = 3
FINETUNE_EPOCHS = 1
FINETUNE_LR = 0.001
SWEEP_BATCH_SIZE = train_state_dict["config"].get("batch_size", 4)
# SWEEP_BATCH_SIZE = 32
SWEEP_ACCUMULATION_STEPS = train_state_dict["config"].get("accumulation_steps", 1)
# SWEEP_ACCUMULATION_STEPS = 32
SWEEP_SEED = train_state_dict["config"].get("seed", 0)
SWEEP_WORKERS = 4
SWEEP_OUTPUT_DIR = Path("runs/vgg16-channel-pruning-sweep")

# Override SWEEP_DATA_ROOT with an absolute path if this selects the wrong copy.
checkpoint_data_root = Path(train_state_dict["config"]["data_root"])
SWEEP_DATA_ROOT = next(
    (root / checkpoint_data_root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / checkpoint_data_root).is_dir()),
    Path.cwd() / checkpoint_data_root,
).resolve()
print(f"Ratios: {SWEEP_RATIOS}")
print(f"Dataset: {SWEEP_DATA_ROOT}")
print(f"Device: {next(model.parameters()).device}; fine-tuning epochs: {FINETUNE_EPOCHS}")
print(f"Results: {SWEEP_OUTPUT_DIR.resolve()}")


Ratios: [0.2, 0.3, 0.4, 0.5]
Dataset: /home/mattgpu/ece690dF26_mrc_project/misc/data/imagenet
Device: cuda:0; fine-tuning epochs: 1
Results: /home/mattgpu/ece690dF26_mrc_project/misc/runs/vgg16-channel-pruning-sweep


In [9]:
import csv
import json
import random
from types import SimpleNamespace
from torch.utils.data import DataLoader
from vgg16_model_train import build_datasets, run_epoch

if not SWEEP_RATIOS or any(not 0 <= ratio < 1 for ratio in SWEEP_RATIOS):
    raise ValueError("Provide at least one pruning ratio in [0, 1)")
if FINETUNE_EPOCHS < 1 or FINETUNE_LR <= 0:
    raise ValueError("Fine-tuning epochs and learning rate must be positive")
if SWEEP_BATCH_SIZE < 1 or SWEEP_ACCUMULATION_STEPS < 1 or SWEEP_WORKERS < 0:
    raise ValueError("Invalid batch size, accumulation steps, or worker count")

sweep_args = SimpleNamespace(**train_state_dict["config"])
sweep_args.data_root = Path(SWEEP_DATA_ROOT)
sweep_args.download = False
training_data, validation_data, classes = build_datasets(sweep_args)
if classes != train_state_dict["classes"]:
    raise ValueError("Dataset class order differs from the checkpoint")
sweep_device = next(model.parameters()).device


conv_indices = [i for i, layer in enumerate(model.features) if isinstance(layer, nn.Conv2d)]
if PRUNE_FEATURE_INDEX not in conv_indices[:-1]:
    raise ValueError("Choose a convolution with a following convolution")
sweep_next_index = conv_indices[conv_indices.index(PRUNE_FEATURE_INDEX) + 1]
original_channels = model.features[sweep_next_index].in_channels
if any(round(ratio * original_channels) >= original_channels for ratio in SWEEP_RATIOS):
    raise ValueError("Every ratio must leave at least one channel")
val_loader = DataLoader(
    validation_data, batch_size=SWEEP_BATCH_SIZE, shuffle=False,
    num_workers=SWEEP_WORKERS, generator=torch.Generator().manual_seed(SWEEP_SEED),
    pin_memory=(sweep_device.type == "cuda"),
    persistent_workers=(SWEEP_WORKERS > 0),
)



print(f"Evaluating baseline on {len(validation_data)} validation samples...", flush=True)
sweep_baseline = run_epoch(model, val_loader, sweep_device)
sweep_results = []
sweep_settings = {
    "checkpoint": str(MODEL_PATH), "training_config": train_state_dict["config"],
    "data_root": str(SWEEP_DATA_ROOT), "ratios": SWEEP_RATIOS,
    "feature_index": PRUNE_FEATURE_INDEX, "next_feature_index": sweep_next_index,
    "finetune_epochs": FINETUNE_EPOCHS, "finetune_lr": FINETUNE_LR,
    "batch_size": SWEEP_BATCH_SIZE, "accumulation_steps": SWEEP_ACCUMULATION_STEPS,
    "seed": SWEEP_SEED, "workers": SWEEP_WORKERS, "device": str(sweep_device),
    "train_samples": len(training_data), "validation_samples": len(validation_data),
}
SWEEP_OUTPUT_DIR = Path(SWEEP_OUTPUT_DIR)
SWEEP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def save_sweep_results():
    report = {"settings": sweep_settings, "baseline": sweep_baseline, "results": sweep_results}
    (SWEEP_OUTPUT_DIR / "results.json").write_text(json.dumps(report, indent=2) + "\n")
    fields = ["ratio", "actual_ratio", "input_channels", "im2col_rows", "parameters",
              "baseline_top1", "pruned_top1", "finetuned_top1", "top1_change_pp",
              "pruned_top5", "finetuned_top5", "pruned_loss", "finetuned_loss"]
    with (SWEEP_OUTPUT_DIR / "results.csv").open("w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(sweep_results)


save_sweep_results()
for ratio in SWEEP_RATIOS:
    print(f"\nPruning ratio {ratio:.2f}: features[{PRUNE_FEATURE_INDEX}] -> "
          f"features[{sweep_next_index}]", flush=True)
    # Always start from the original weights, never the previous trial.
    candidate, reference = prune_conv_pair(model, ratio, PRUNE_FEATURE_INDEX)
    del reference
    random.seed(SWEEP_SEED)
    torch.manual_seed(SWEEP_SEED)
    train_loader = DataLoader(
        training_data, batch_size=SWEEP_BATCH_SIZE, shuffle=True,
        num_workers=SWEEP_WORKERS, generator=torch.Generator().manual_seed(SWEEP_SEED),
        pin_memory=(sweep_device.type == "cuda"),
        persistent_workers=(SWEEP_WORKERS > 0),
    )
    immediate = run_epoch(candidate, val_loader, sweep_device)
    # Rebuilt layers have new Parameters, so the optimizer must be created here.
    optimizer = torch.optim.SGD(candidate.parameters(), lr=FINETUNE_LR,
                                momentum=sweep_args.momentum,
                                weight_decay=sweep_args.weight_decay)
    history = []
    for epoch in range(FINETUNE_EPOCHS):
        print(f"Ratio {ratio:.2f}, fine-tuning epoch {epoch + 1}/{FINETUNE_EPOCHS}", flush=True)
        training = run_epoch(candidate, train_loader, sweep_device, optimizer,
                             accumulation_steps=SWEEP_ACCUMULATION_STEPS)
        recovered = run_epoch(candidate, val_loader, sweep_device)
        history.append({"epoch": epoch + 1, "train": training, "val": recovered})
    conv = candidate.features[sweep_next_index]
    sweep_results.append({
        "ratio": ratio, "actual_ratio": 1 - conv.in_channels / original_channels,
        "input_channels": conv.in_channels,
        "im2col_rows": conv.in_channels * conv.kernel_size[0] * conv.kernel_size[1],
        "parameters": sum(p.numel() for p in candidate.parameters()),
        "baseline_top1": sweep_baseline["top1"], "pruned_top1": immediate["top1"],
        "finetuned_top1": recovered["top1"],
        "top1_change_pp": recovered["top1"] - sweep_baseline["top1"],
        "pruned_top5": immediate["top5"], "finetuned_top5": recovered["top5"],
        "pruned_loss": immediate["loss"], "finetuned_loss": recovered["loss"],
        "history": history,
    })
    save_sweep_results()
    print(f"Saved ratio {ratio:.2f}: top-1 {immediate['top1']:.2f}% -> "
          f"{recovered['top1']:.2f}% (baseline {sweep_baseline['top1']:.2f}%)", flush=True)
    del optimizer, candidate, train_loader

print("\n ratio  channels  im2col rows  baseline top1  pruned top1  finetuned top1  change (pp)")
for row in sweep_results:
    print(f" {row['ratio']:.2f}   {row['input_channels']:7d}  {row['im2col_rows']:11d}"
          f"  {row['baseline_top1']:13.2f}  {row['pruned_top1']:11.2f}"
          f"  {row['finetuned_top1']:14.2f}  {row['top1_change_pp']:+11.2f}")
print(f"Results saved to {SWEEP_OUTPUT_DIR.resolve()}")


Evaluating baseline on 5000 validation samples...
val 100/1250 loss=2.4646
val 200/1250 loss=2.5756
val 300/1250 loss=2.5610
val 400/1250 loss=2.5489
val 500/1250 loss=2.5351
val 600/1250 loss=2.5286
val 700/1250 loss=2.5050
val 800/1250 loss=2.5026
val 900/1250 loss=2.5109
val 1000/1250 loss=2.5108
val 1100/1250 loss=2.5028
val 1200/1250 loss=2.5168
val 1250/1250 loss=2.5160

Pruning ratio 0.20: features[0] -> features[2]
val 100/1250 loss=2.4820
val 200/1250 loss=2.5950
val 300/1250 loss=2.5768
val 400/1250 loss=2.5695
val 500/1250 loss=2.5535
val 600/1250 loss=2.5489
val 700/1250 loss=2.5236
val 800/1250 loss=2.5229
val 900/1250 loss=2.5310
val 1000/1250 loss=2.5301
val 1100/1250 loss=2.5237
val 1200/1250 loss=2.5360
val 1250/1250 loss=2.5350
Ratio 0.20, fine-tuning epoch 1/1
train 100/11250 loss=2.8045
train 200/11250 loss=2.9102
train 300/11250 loss=2.8545
train 400/11250 loss=2.8740
train 500/11250 loss=2.8812
train 600/11250 loss=2.9057
train 700/11250 loss=2.9180
train 800/1125

### Extending the experiment

Use the table and saved CSV/JSON to compare validation accuracy with actual
channel counts, im2col rows, and parameter counts. Channel counts are rounded,
so `actual_ratio` can differ from the requested ratio. Choose ratios using
validation accuracy and cost; reserve the official test set for final evaluation.

To measure the existing estimator separately, pass the compact VGG directly:

```python
before = model_to_he_im2col_ops(model, input_shape=tuple(example.shape))
after = model_to_he_im2col_ops(l1_m, input_shape=tuple(example.shape))
name = f"features.{next_index}"
print(before["layers"][name]["packing"]["input_ciphertexts"],
      after["layers"][name]["packing"]["input_ciphertexts"])
```

Here `l1_m` is the single-pair demo from cell 8; the sweep saves metrics rather
than model checkpoints. To prune multiple intermediate pairs, call
`prune_conv_pair` successively on a compact model, skipping the final convolution.
That is a separate experiment: layer interactions and pruning order can affect
the outcome. Saving a compact `state_dict` also requires recording the changed
architecture/channel indices; it cannot load into an unchanged `vgg16()`.
